In [1]:
# CELL CONFIG - chạy cell này đầu tiên.
# Sửa trực tiếp 3 đường dẫn bên dưới nếu thư mục Google Drive của bạn khác.

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Không chạy trên Colab hoặc Drive chưa được mount:', e)

from pathlib import Path

ASL_ROOT = '/content/drive/MyDrive/data/ASL'
VSL_ROOT = '/content/drive/MyDrive/data/VSL'
OUTPUT_DIR = '/content/drive/MyDrive/data/output_4.7'

if not ASL_ROOT or not VSL_ROOT or not OUTPUT_DIR:
    raise ValueError('Bạn cần nhập đủ ASL_ROOT, VSL_ROOT và OUTPUT_DIR.')

CONFIG = {
    'DATASETS': [
        {'name': 'ASL', 'root': ASL_ROOT, 'label_map': ''},
        {'name': 'VSL', 'root': VSL_ROOT, 'label_map': ''},
    ],
    'OUTPUT_DIR': OUTPUT_DIR,
    'NPZ_GLOB': '**/*.npz',
    'FPS': None,
    'LANDMARK_LAYOUT': 'asl_vsl_extractor_v2',
    'USE_POSE_WRISTS_FOR_HAND_CENTERS': True,
    'ASSUME_HANDS_ARE_LAST_126_DIMS': True,
    'INVALID_ZERO_POINTS': True,
    'INTERPOLATE_MISSING_CENTERS': True,
    'MAX_INTERPOLATION_GAP': 8,
    'NORMALIZE_BY_BODY_ANCHOR': True,
    'MIN_VALID_HAND_FRAMES': 3,
    'RUN_DTW': True,
    'DTW_RESAMPLE_LEN': 30,
    'DTW_WINDOW': 8,
    'DTW_MIN_SAMPLES_PER_LABEL': 2,
    'DTW_MAX_PAIRS_PER_LABEL': None,
    'DTW_RANDOM_SEED': 42,
    'RUN_INTER_CLASS_DTW': True,
    'INTER_CLASS_DTW_EXHAUSTIVE': False,
    'INTER_CLASS_CANDIDATES_PER_LABEL': 20,
    'INTER_CLASS_MAX_EXHAUSTIVE_LABELS': 600,
    'SAVE_TRAJECTORY_CACHE': True,
    'PLOT_TOP_N_LABELS': 30,
    'TRAJECTORY_EXAMPLES_PER_DATASET': 6,
    'CHECKPOINT_EVERY': 500,
    'FORCE_RECOMPUTE': True,
}

Path(CONFIG['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
CONFIG


Mounted at /content/drive


{'DATASETS': [{'name': 'ASL',
   'root': '/content/drive/MyDrive/data/ASL',
   'label_map': ''},
  {'name': 'VSL', 'root': '/content/drive/MyDrive/data/VSL', 'label_map': ''}],
 'OUTPUT_DIR': '/content/drive/MyDrive/data/output_4.7',
 'NPZ_GLOB': '**/*.npz',
 'FPS': None,
 'LANDMARK_LAYOUT': 'asl_vsl_extractor_v2',
 'USE_POSE_WRISTS_FOR_HAND_CENTERS': True,
 'ASSUME_HANDS_ARE_LAST_126_DIMS': True,
 'INVALID_ZERO_POINTS': True,
 'INTERPOLATE_MISSING_CENTERS': True,
 'MAX_INTERPOLATION_GAP': 8,
 'NORMALIZE_BY_BODY_ANCHOR': True,
 'MIN_VALID_HAND_FRAMES': 3,
 'RUN_DTW': True,
 'DTW_RESAMPLE_LEN': 30,
 'DTW_WINDOW': 8,
 'DTW_MIN_SAMPLES_PER_LABEL': 2,
 'DTW_MAX_PAIRS_PER_LABEL': None,
 'DTW_RANDOM_SEED': 42,
 'RUN_INTER_CLASS_DTW': True,
 'INTER_CLASS_DTW_EXHAUSTIVE': False,
 'INTER_CLASS_CANDIDATES_PER_LABEL': 20,
 'INTER_CLASS_MAX_EXHAUSTIVE_LABELS': 600,
 'SAVE_TRAJECTORY_CACHE': True,
 'PLOT_TOP_N_LABELS': 30,
 'TRAJECTORY_EXAMPLES_PER_DATASET': 6,
 'CHECKPOINT_EVERY': 500,
 'FORCE_REC

# 4.7. Phân tích đặc trưng chuyển động ASL/VSL

Notebook này chạy trực tiếp trên Colab bằng các cell Python, không sinh file `.py` trung gian. Dữ liệu đầu vào là các file `.npz` do pipeline trích xuất ASL/VSL hiện tại tạo ra, gồm `pose`, `hands`, `face`, `mouth`, `valid_mask`, `label`, `video_name`, `source_fps`, `target_frames`.

## 1. Nạp thư viện, config mặc định và helper đọc format ASL/VSL

In [2]:
# -*- coding: utf-8 -*-
"""
Section 4.7 - Motion Feature Analysis for ASL/VSL extractor .npz datasets.

What this script does:
1) Recursively scans ASL and VSL .npz files.
2) Reads the current ASL/VSL extractor format plus older flat MediaPipe sequence formats.
3) Extracts motion features per video/sample.
4) Summarizes results by label and by dataset.
5) Generates plots.
6) Runs DTW analysis: intra-class DTW and inter-class candidate DTW.

Expected use:
    python run_4_7_motion_analysis.py --config config_4_7.json

Edit config_4_7.json with your real dataset paths when .npz files are available.
"""

from __future__ import annotations

import csv
import gc
import json
import math
import os
import re
import sys
import traceback
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

import matplotlib.pyplot as plt


# =========================
# Default configuration
# =========================

DEFAULT_CONFIG: Dict[str, Any] = {
    "DATASETS": [
        {
            "name": "ASL",
            "root": "",
            "label_map": "",
        },
        {
            "name": "VSL",
            "root": "",
            "label_map": "",
        },
    ],
    "OUTPUT_DIR": "",
    "NPZ_GLOB": "**/*.npz",

    # Frame rate is optional. If FPS is None, velocity is measured as displacement per frame.
    "FPS": None,

    # Keypoint handling.
    # auto mode assumes hand landmarks are the final 126 flat dimensions when explicit hand arrays are unavailable.
    # This supports common MediaPipe concatenations such as:
    # - pose(33*3)+face(468*3)+left_hand(21*3)+right_hand(21*3)=1629
    # - pose(33*4)+face(468*3)+left_hand(21*3)+right_hand(21*3)=1662
    # - pose(25*3)+face(468*3)+left_hand(21*3)+right_hand(21*3)=1605
    "LANDMARK_LAYOUT": "auto",
    "ASSUME_HANDS_ARE_LAST_126_DIMS": True,
    "INVALID_ZERO_POINTS": True,
    "INTERPOLATE_MISSING_CENTERS": True,
    "MAX_INTERPOLATION_GAP": 8,
    "NORMALIZE_BY_BODY_ANCHOR": True,
    "MIN_VALID_HAND_FRAMES": 3,
    "USE_POSE_WRISTS_FOR_HAND_CENTERS": True,

    # DTW settings.
    "RUN_DTW": True,
    "DTW_RESAMPLE_LEN": 30,
    "DTW_WINDOW": 8,
    "DTW_MIN_SAMPLES_PER_LABEL": 2,
    "DTW_MAX_PAIRS_PER_LABEL": None,  # None = all intra-class pairs.
    "DTW_RANDOM_SEED": 42,

    # Inter-class DTW: full exhaustive all label pairs can be extremely slow for thousands of classes.
    # Default mode still uses ALL labels, but first finds top nearest candidate labels by vectorized Euclidean distance,
    # then computes DTW only for those likely-confusing candidates.
    "RUN_INTER_CLASS_DTW": True,
    "INTER_CLASS_DTW_EXHAUSTIVE": False,
    "INTER_CLASS_CANDIDATES_PER_LABEL": 20,
    "INTER_CLASS_MAX_EXHAUSTIVE_LABELS": 600,

    # Plot / output options.
    "SAVE_TRAJECTORY_CACHE": True,
    "PLOT_TOP_N_LABELS": 30,
    "TRAJECTORY_EXAMPLES_PER_DATASET": 6,
    "CHECKPOINT_EVERY": 500,
    "FORCE_RECOMPUTE": True,
}

# ASL/VSL extractor schema constants. The extractor stores one .npz per video with
# pose, hands, face blendshape, mouth, and valid_mask arrays instead of a single
# old-style keypoints tensor.
POSE_LANDMARK_COUNT = 33
HAND_LANDMARK_COUNT = 21
POSE_COORD_DIM = 5
POSE_NORM_DIM = POSE_LANDMARK_COUNT * POSE_COORD_DIM
POSE_WORLD_DIM = POSE_LANDMARK_COUNT * POSE_COORD_DIM
POSE_FEATURE_DIM = POSE_NORM_DIM + POSE_WORLD_DIM
HAND_COORD_DIM = 3
LEFT_HAND_NORM_DIM = HAND_LANDMARK_COUNT * HAND_COORD_DIM
RIGHT_HAND_NORM_DIM = HAND_LANDMARK_COUNT * HAND_COORD_DIM
LEFT_HAND_WORLD_DIM = HAND_LANDMARK_COUNT * HAND_COORD_DIM
RIGHT_HAND_WORLD_DIM = HAND_LANDMARK_COUNT * HAND_COORD_DIM
HANDS_FEATURE_DIM = LEFT_HAND_NORM_DIM + RIGHT_HAND_NORM_DIM + LEFT_HAND_WORLD_DIM + RIGHT_HAND_WORLD_DIM
VALID_MASK_COLUMNS = ("pose", "left_hand", "right_hand", "face")
POSE_HEAD_LANDMARKS = tuple(range(0, 11))
POSE_LEFT_WRIST_INDEX = 15
POSE_RIGHT_WRIST_INDEX = 16




# =========================
# Utility functions
# =========================

def load_config(config: Optional[Any]) -> Dict[str, Any]:
    cfg = json.loads(json.dumps(DEFAULT_CONFIG))
    if isinstance(config, dict):
        cfg.update(config)
    elif config:
        with open(config, "r", encoding="utf-8") as f:
            user_cfg = json.load(f)
        cfg.update(user_cfg)
    return cfg


def ensure_dirs(output_dir: Path) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    dirs = {
        "root": output_dir,
        "figures": output_dir / "figures",
        "dtw": output_dir / "dtw",
        "debug": output_dir / "debug",
        "cache": output_dir / "cache",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def safe_float(x: Any) -> float:
    try:
        v = float(x)
        if math.isfinite(v):
            return v
    except Exception:
        pass
    return float("nan")


def safe_mean(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if arr.size else float("nan")


def safe_std(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.std(arr)) if arr.size else float("nan")


def safe_min(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.min(arr)) if arr.size else float("nan")


def safe_max(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(arr)) if arr.size else float("nan")


def safe_median(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if arr.size else float("nan")


def flatten_keypoint_array(arr: np.ndarray) -> Optional[np.ndarray]:
    """Convert common sequence shapes to (N,T,D)."""
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.dtype == object:
        try:
            arr = np.stack(arr.tolist())
        except Exception:
            return None
    if not np.issubdtype(arr.dtype, np.number):
        return None
    arr = arr.astype(np.float32, copy=False)

    if arr.ndim == 1:
        return None
    if arr.ndim == 2:
        # (T,D)
        return arr[None, :, :]
    if arr.ndim == 3:
        # Either (N,T,D) or (T,K,C)
        if arr.shape[-1] in (2, 3, 4) and arr.shape[1] > 10:
            # (T,K,C) -> (1,T,K*C)
            t, k, c = arr.shape
            return arr.reshape(1, t, k * c)
        else:
            return arr
    if arr.ndim == 4:
        # (N,T,K,C) -> (N,T,K*C)
        n, t, k, c = arr.shape
        return arr.reshape(n, t, k * c)
    return None


def as_numeric_2d(arr: Any) -> Optional[np.ndarray]:
    """Return a numeric (T,D) array, or None for scalar/non-sequence values."""
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.dtype == object:
        try:
            arr = np.stack(arr.tolist())
        except Exception:
            return None
    if not np.issubdtype(arr.dtype, np.number):
        return None
    arr = arr.astype(np.float32, copy=False)
    if arr.ndim == 2 and arr.shape[0] > 0 and arr.shape[1] > 0:
        return arr
    if arr.ndim == 3 and arr.shape[0] == 1 and arr.shape[1] > 0 and arr.shape[2] > 0:
        return arr[0]
    return None


def scalar_from_npz(npz: Any, key: str, default: Any = None) -> Any:
    if key not in npz.keys():
        return default
    try:
        arr = np.asarray(npz[key])
        if arr.ndim == 0:
            value = arr.item()
        elif arr.size:
            value = arr.flat[0]
        else:
            return default
        if isinstance(value, bytes):
            return value.decode("utf-8", errors="replace")
        if isinstance(value, np.generic):
            return value.item()
        return value
    except Exception:
        return default


def is_asl_vsl_extraction_npz(npz: Any) -> bool:
    keys = set(npz.keys())
    return "hands" in keys and ("pose" in keys or "mouth" in keys or "valid_mask" in keys)


def build_asl_vsl_sequence_array(npz: Any) -> Optional[np.ndarray]:
    if not is_asl_vsl_extraction_npz(npz):
        return None
    arrays: List[np.ndarray] = []
    target_t: Optional[int] = None
    for key in ["pose", "hands", "face", "mouth"]:
        if key not in npz.keys():
            continue
        arr = as_numeric_2d(npz[key])
        if arr is None:
            continue
        if target_t is None:
            target_t = int(arr.shape[0])
        if int(arr.shape[0]) != target_t:
            continue
        arrays.append(arr)
    if not arrays:
        return None
    return np.concatenate(arrays, axis=1)[None, :, :]


def find_sequence_array(npz: Any) -> Tuple[Optional[str], Optional[np.ndarray]]:
    # Current repo extractor format: one video per .npz, with separate pose/hands/face/mouth arrays.
    repo_seq = build_asl_vsl_sequence_array(npz)
    if repo_seq is not None:
        return "pose+hands+face+mouth", repo_seq

    preferred = [
        "sequences", "sequence", "keypoints", "keypoint", "data", "X", "x",
        "arr_0", "features", "landmarks", "inputs", "input"
    ]
    keys = list(npz.keys())
    for key in preferred:
        if key in keys:
            arr = flatten_keypoint_array(npz[key])
            if arr is not None:
                return key, arr
    # fallback: largest numeric array that can become sequence
    best_key, best_arr, best_size = None, None, -1
    for key in keys:
        try:
            arr = flatten_keypoint_array(npz[key])
            if arr is not None and arr.size > best_size:
                best_key, best_arr, best_size = key, arr, arr.size
        except Exception:
            continue
    return best_key, best_arr


def normalize_explicit_part(arr: Optional[np.ndarray]) -> Optional[np.ndarray]:
    """Return explicit landmark array as (N,T,K,C), where C is first 3 coordinates when possible."""
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.dtype == object:
        try:
            arr = np.stack(arr.tolist())
        except Exception:
            return None
    if not np.issubdtype(arr.dtype, np.number):
        return None
    arr = arr.astype(np.float32, copy=False)
    if arr.ndim == 2:
        # Keep old behavior for ambiguous flat arrays; ASL/VSL extractor arrays are handled separately below.
        return None
    if arr.ndim == 3:
        # (T,K,C) -> (1,T,K,C)
        if arr.shape[-1] >= 2 and arr.shape[1] >= 1:
            return arr[None, :, :, :min(arr.shape[-1], 3)]
    if arr.ndim == 4:
        return arr[:, :, :, :min(arr.shape[-1], 3)]
    return None


def get_valid_mask(npz: Any, n_frames: Optional[int] = None) -> Optional[np.ndarray]:
    if "valid_mask" not in npz.keys():
        return None
    arr = np.asarray(npz["valid_mask"])
    if arr.ndim != 2 or arr.shape[1] < len(VALID_MASK_COLUMNS):
        return None
    if n_frames is not None and arr.shape[0] != n_frames:
        return None
    return arr[:, :len(VALID_MASK_COLUMNS)].astype(bool)


def apply_frame_mask(lm: Optional[np.ndarray], valid: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if lm is None or valid is None:
        return lm
    if lm.shape[0] != len(valid):
        return lm
    out = lm.astype(np.float32, copy=True)
    out[~valid] = np.nan
    return out


def reshape_asl_vsl_pose(pose_arr: Any) -> Optional[np.ndarray]:
    arr = as_numeric_2d(pose_arr)
    if arr is None:
        return None
    if arr.shape[1] >= POSE_NORM_DIM:
        block = arr[:, :POSE_NORM_DIM]
        return block.reshape(arr.shape[0], POSE_LANDMARK_COUNT, POSE_COORD_DIM)[:, :, :3]
    return reshape_pose_block(arr)


def reshape_asl_vsl_hand(hands_arr: Any, side: str) -> Optional[np.ndarray]:
    arr = as_numeric_2d(hands_arr)
    if arr is None:
        return None
    if arr.shape[1] < LEFT_HAND_NORM_DIM + RIGHT_HAND_NORM_DIM:
        return None
    if side == "left":
        block = arr[:, 0:LEFT_HAND_NORM_DIM]
    else:
        block = arr[:, LEFT_HAND_NORM_DIM:LEFT_HAND_NORM_DIM + RIGHT_HAND_NORM_DIM]
    return block.reshape(arr.shape[0], HAND_LANDMARK_COUNT, HAND_COORD_DIM)[:, :, :3]


def reshape_asl_vsl_mouth(mouth_arr: Any) -> Optional[np.ndarray]:
    arr = as_numeric_2d(mouth_arr)
    if arr is None or arr.shape[1] < 6 or arr.shape[1] % 3 != 0:
        return None
    return arr.reshape(arr.shape[0], arr.shape[1] // 3, 3)[:, :, :3]


def hand_proxy_from_pose_wrist(pose_lm: Optional[np.ndarray], side: str) -> Optional[np.ndarray]:
    if pose_lm is None:
        return None
    wrist_idx = POSE_LEFT_WRIST_INDEX if side == "left" else POSE_RIGHT_WRIST_INDEX
    if pose_lm.ndim != 3 or pose_lm.shape[1] <= wrist_idx:
        return None
    wrist = pose_lm[:, wrist_idx, :3]
    return np.repeat(wrist[:, None, :], HAND_LANDMARK_COUNT, axis=1).astype(np.float32)


def face_proxy_from_pose_head(pose_lm: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if pose_lm is None or pose_lm.ndim != 3:
        return None
    idx = [i for i in POSE_HEAD_LANDMARKS if i < pose_lm.shape[1]]
    if not idx:
        return None
    return pose_lm[:, idx, :3].astype(np.float32)


def get_asl_vsl_extraction_parts(npz: Any, cfg: Dict[str, Any]) -> Dict[str, Optional[np.ndarray]]:
    pose_lm = reshape_asl_vsl_pose(npz["pose"]) if "pose" in npz.keys() else None
    n_frames = int(pose_lm.shape[0]) if pose_lm is not None else None
    valid_mask = get_valid_mask(npz, n_frames)

    if pose_lm is not None and valid_mask is not None:
        pose_lm = apply_frame_mask(pose_lm, valid_mask[:, 0])

    use_pose_wrists = bool(cfg.get("USE_POSE_WRISTS_FOR_HAND_CENTERS", True))
    left_hand = hand_proxy_from_pose_wrist(pose_lm, "left") if use_pose_wrists else None
    right_hand = hand_proxy_from_pose_wrist(pose_lm, "right") if use_pose_wrists else None

    if left_hand is None and "hands" in npz.keys():
        left_hand = reshape_asl_vsl_hand(npz["hands"], "left")
    if right_hand is None and "hands" in npz.keys():
        right_hand = reshape_asl_vsl_hand(npz["hands"], "right")

    if valid_mask is not None:
        left_hand = apply_frame_mask(left_hand, valid_mask[:, 1])
        right_hand = apply_frame_mask(right_hand, valid_mask[:, 2])

    face_lm = face_proxy_from_pose_head(pose_lm)
    if face_lm is None and "mouth" in npz.keys():
        face_lm = reshape_asl_vsl_mouth(npz["mouth"])
        if valid_mask is not None:
            face_lm = apply_frame_mask(face_lm, valid_mask[:, 3])

    return {
        "left_hand": left_hand,
        "right_hand": right_hand,
        "pose": pose_lm,
        "face": face_lm,
    }


def get_explicit_parts(npz: Any, cfg: Dict[str, Any]) -> Dict[str, Optional[np.ndarray]]:
    if is_asl_vsl_extraction_npz(npz):
        parts = get_asl_vsl_extraction_parts(npz, cfg)
        if any(v is not None for v in parts.values()):
            return parts

    key_aliases = {
        "left_hand": ["left_hand", "lefthand", "lh", "left", "left_hand_landmarks", "leftHand"],
        "right_hand": ["right_hand", "righthand", "rh", "right", "right_hand_landmarks", "rightHand"],
        "pose": ["pose", "pose_landmarks", "body", "body_pose", "skeleton"],
        "face": ["face", "face_landmarks", "facemesh"],
    }
    keys_lower = {k.lower(): k for k in npz.keys()}
    parts: Dict[str, Optional[np.ndarray]] = {}
    for part, aliases in key_aliases.items():
        found = None
        for alias in aliases:
            if alias.lower() in keys_lower:
                found = keys_lower[alias.lower()]
                break
        parts[part] = normalize_explicit_part(npz[found]) if found else None
    return parts


def extraction_metadata_from_npz(npz: Any) -> Dict[str, Any]:
    meta: Dict[str, Any] = {}
    if is_asl_vsl_extraction_npz(npz):
        meta["npz_format"] = "asl_vsl_extractor_v2"

    for key in ["video_name", "target_frames", "source_fps", "source_total_frames", "train_feature_dim"]:
        value = scalar_from_npz(npz, key, None)
        if value is not None:
            meta[key] = value

    valid_mask = get_valid_mask(npz)
    if valid_mask is not None and valid_mask.size:
        ratios = valid_mask.mean(axis=0)
        for idx, name in enumerate(VALID_MASK_COLUMNS):
            meta[f"valid_{name}_ratio"] = float(ratios[idx])
    return meta


def pick_sample_part(part: Optional[np.ndarray], idx: int, n_samples: int) -> Optional[np.ndarray]:
    if part is None:
        return None
    # Current ASL/VSL extractor format stores one video per .npz, so explicit
    # pose/hand/face parts are already shaped (T,K,C). Older multi-sample
    # bundles use (N,T,K,C).
    if part.ndim == 3:
        return part
    if part.ndim != 4:
        return None
    if part.shape[0] == n_samples:
        return part[idx]
    if part.shape[0] == 1:
        return part[0]
    if idx < part.shape[0]:
        return part[idx]
    return None


def infer_label_from_path(path: Path, root: Path, sample_idx: int, label_map: Dict[str, str]) -> str:
    # Priority: parent directory if class-like, else file stem.
    try:
        rel = path.relative_to(root)
        parts = rel.parts
    except Exception:
        parts = path.parts
    candidates = []
    if len(parts) >= 2:
        candidates.append(parts[-2])
    candidates.append(path.stem)
    if sample_idx is not None:
        candidates.append(f"{path.stem}_{sample_idx}")

    for c in candidates:
        if c in label_map:
            return str(label_map[c])
    # If filename has class_0001 anywhere.
    m = re.search(r"class_\d+", path.as_posix())
    if m and m.group(0) in label_map:
        return str(label_map[m.group(0)])
    return str(candidates[0])


def load_label_map(path: str) -> Dict[str, str]:
    if not path:
        return {}
    p = Path(path)
    if not p.exists():
        print(f"[WARN] label_map not found: {p}")
        return {}
    try:
        if p.suffix.lower() == ".json":
            data = json.load(open(p, "r", encoding="utf-8"))
            if isinstance(data, dict):
                return {str(k): str(v) for k, v in data.items()}
            if isinstance(data, list):
                return {str(i): str(v) for i, v in enumerate(data)}
        if p.suffix.lower() == ".csv":
            df = pd.read_csv(p)
            cols = list(df.columns)
            if len(cols) >= 2:
                return {str(r[cols[0]]): str(r[cols[1]]) for _, r in df.iterrows()}
        # txt: line can be "0 gloss" or just gloss
        mp = {}
        with open(p, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                parts = line.split(maxsplit=1)
                if len(parts) == 2 and parts[0].isdigit():
                    mp[str(parts[0])] = parts[1]
                    mp[f"class_{int(parts[0]):04d}"] = parts[1]
                else:
                    mp[str(i)] = line
        return mp
    except Exception as e:
        print(f"[WARN] cannot load label_map {p}: {e}")
        return {}


def labels_from_npz(npz: Any, n_samples: int, path: Path, root: Path, label_map: Dict[str, str]) -> List[str]:
    label_keys = ["labels", "label", "y", "Y", "classes", "class", "target", "targets"]
    raw = None
    for key in label_keys:
        if key in npz.keys():
            raw = npz[key]
            break
    labels = []
    if raw is not None:
        raw_arr = np.asarray(raw)
        if raw_arr.ndim == 0:
            raw_values = [raw_arr.item()] * n_samples
        elif raw_arr.shape[0] == n_samples:
            raw_values = raw_arr.tolist()
        else:
            raw_values = [raw_arr.flat[0]] * n_samples if raw_arr.size else [None] * n_samples
        for value in raw_values:
            key = str(value)
            # numeric labels may be float-like
            if re.fullmatch(r"\d+\.0", key):
                key = key.split(".")[0]
            label = label_map.get(key, label_map.get(f"class_{int(key):04d}", key) if key.isdigit() else key)
            labels.append(str(label))
        return labels
    return [infer_label_from_path(path, root, i, label_map) for i in range(n_samples)]


def is_valid_point(points: np.ndarray, invalid_zero: bool = True) -> np.ndarray:
    # points: (..., C)
    finite = np.all(np.isfinite(points), axis=-1)
    if invalid_zero:
        nonzero = np.linalg.norm(np.nan_to_num(points[..., :2], nan=0.0), axis=-1) > 1e-8
        return finite & nonzero
    return finite


def reshape_known_block(block: np.ndarray, n_landmarks: int, coord_dim: int = 3) -> Optional[np.ndarray]:
    # block: (T, n_landmarks*coord_dim)
    if block is None:
        return None
    if block.ndim == 3:
        return block[:, :, :3]
    if block.ndim != 2:
        return None
    T, D = block.shape
    needed = n_landmarks * coord_dim
    if D < needed:
        return None
    return block[:, :needed].reshape(T, n_landmarks, coord_dim)[:, :, :3]


def reshape_pose_block(block: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if block is None:
        return None
    if block.ndim == 3:
        return block[:, :, :3]
    if block.ndim != 2:
        return None
    T, D = block.shape
    for coord_dim in (4, 3):
        if D % coord_dim == 0:
            k = D // coord_dim
            if 10 <= k <= 100:
                return block.reshape(T, k, coord_dim)[:, :, :3]
    return None


def infer_parts_from_flat(seq: np.ndarray, cfg: Dict[str, Any]) -> Dict[str, Optional[np.ndarray]]:
    """Infer pose/face/hand arrays from flat sequence (T,D)."""
    T, D = seq.shape
    parts: Dict[str, Optional[np.ndarray]] = {"left_hand": None, "right_hand": None, "pose": None, "face": None}

    if cfg.get("ASSUME_HANDS_ARE_LAST_126_DIMS", True) and D >= 126:
        hands_start = D - 126
        left_block = seq[:, hands_start:hands_start + 63]
        right_block = seq[:, hands_start + 63:hands_start + 126]
        parts["left_hand"] = reshape_known_block(left_block, 21, 3)
        parts["right_hand"] = reshape_known_block(right_block, 21, 3)

        # Try face immediately before hands: 468 * 3 = 1404 dims.
        face_dim = 468 * 3
        face_start = hands_start - face_dim
        if face_start >= 0:
            face_block = seq[:, face_start:hands_start]
            parts["face"] = reshape_known_block(face_block, 468, 3)
            pose_block = seq[:, :face_start]
            parts["pose"] = reshape_pose_block(pose_block)
        else:
            pose_block = seq[:, :hands_start]
            parts["pose"] = reshape_pose_block(pose_block)
    else:
        # Last-resort: if total dim is exactly 42 landmarks * 3.
        if D == 126:
            parts["left_hand"] = seq[:, :63].reshape(T, 21, 3)
            parts["right_hand"] = seq[:, 63:126].reshape(T, 21, 3)
    return parts


def center_from_landmarks(lm: Optional[np.ndarray], invalid_zero: bool = True) -> Tuple[Optional[np.ndarray], float, float]:
    """Compute center per frame from landmarks (T,K,3). Return center (T,3), frame_missing_ratio, point_missing_ratio."""
    if lm is None:
        return None, float("nan"), float("nan")
    lm = np.asarray(lm, dtype=np.float32)
    if lm.ndim != 3 or lm.shape[-1] < 2:
        return None, float("nan"), float("nan")
    valid = is_valid_point(lm[..., :3], invalid_zero=invalid_zero)
    point_missing_ratio = 1.0 - float(np.mean(valid)) if valid.size else float("nan")
    valid_counts = valid.sum(axis=1)
    frame_valid = valid_counts > 0
    frame_missing_ratio = 1.0 - float(np.mean(frame_valid)) if frame_valid.size else float("nan")
    center = np.full((lm.shape[0], 3), np.nan, dtype=np.float32)
    for t in range(lm.shape[0]):
        if frame_valid[t]:
            center[t] = np.nanmean(lm[t, valid[t], :3], axis=0)
    return center, frame_missing_ratio, point_missing_ratio


def interpolate_centers(center: Optional[np.ndarray], max_gap: int = 8) -> Optional[np.ndarray]:
    if center is None:
        return None
    c = np.asarray(center, dtype=np.float32).copy()
    T = c.shape[0]
    for dim in range(c.shape[1]):
        y = c[:, dim]
        valid = np.isfinite(y)
        if valid.sum() < 2:
            continue
        x = np.arange(T)
        interp = np.interp(x, x[valid], y[valid]).astype(np.float32)
        # Keep long missing gaps as nan.
        missing = ~valid
        if missing.any() and max_gap is not None:
            start = None
            for i, m in enumerate(missing.tolist() + [False]):
                if m and start is None:
                    start = i
                if not m and start is not None:
                    end = i
                    if end - start <= max_gap:
                        y[start:end] = interp[start:end]
                    start = None
            c[:, dim] = y
        else:
            c[:, dim] = interp
    return c


def robust_body_scale(pose_lm: Optional[np.ndarray], left_center: Optional[np.ndarray], right_center: Optional[np.ndarray], invalid_zero: bool = True) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Return body center per frame and a scalar scale for normalization."""
    body_center = None
    scale = None
    if pose_lm is not None:
        body_center, _, _ = center_from_landmarks(pose_lm, invalid_zero=invalid_zero)
        # Try shoulder distance MediaPipe indexes 11 and 12.
        if pose_lm.shape[1] > 12:
            lsh = pose_lm[:, 11, :3]
            rsh = pose_lm[:, 12, :3]
            valid = is_valid_point(lsh, invalid_zero=invalid_zero) & is_valid_point(rsh, invalid_zero=invalid_zero)
            d = np.linalg.norm(lsh[:, :2] - rsh[:, :2], axis=1)
            d = d[valid & np.isfinite(d) & (d > 1e-6)]
            if d.size:
                scale = float(np.median(d))
    # Fallback scale from spread of available centers.
    if scale is None or not np.isfinite(scale) or scale <= 1e-6:
        pts = []
        for c in [body_center, left_center, right_center]:
            if c is not None:
                pts.append(c[:, :2])
        if pts:
            all_pts = np.vstack(pts)
            all_pts = all_pts[np.all(np.isfinite(all_pts), axis=1)]
            if all_pts.shape[0] > 2:
                ranges = np.nanmax(all_pts, axis=0) - np.nanmin(all_pts, axis=0)
                val = float(np.nanmax(ranges))
                if np.isfinite(val) and val > 1e-6:
                    scale = val
    return body_center, scale


def normalize_center(center: Optional[np.ndarray], body_center: Optional[np.ndarray], scale: Optional[float], enabled: bool) -> Optional[np.ndarray]:
    if center is None:
        return None
    c = center.astype(np.float32).copy()
    if enabled and body_center is not None:
        bc = body_center.astype(np.float32)
        mask = np.all(np.isfinite(bc), axis=1)
        c[mask] = c[mask] - bc[mask]
    if enabled and scale is not None and np.isfinite(scale) and scale > 1e-6:
        c = c / float(scale)
    return c


def consecutive_distances(center: Optional[np.ndarray]) -> np.ndarray:
    if center is None or center.shape[0] < 2:
        return np.array([], dtype=np.float32)
    a = center[1:, :3]
    b = center[:-1, :3]
    valid = np.all(np.isfinite(a), axis=1) & np.all(np.isfinite(b), axis=1)
    d = np.full(center.shape[0] - 1, np.nan, dtype=np.float32)
    d[valid] = np.linalg.norm(a[valid] - b[valid], axis=1)
    return d


def trajectory_stats(center: Optional[np.ndarray]) -> Dict[str, float]:
    d = consecutive_distances(center)
    total = safe_sum(d)
    if center is None:
        return {"path_length": float("nan"), "displacement": float("nan"), "straightness": float("nan"), "direction_changes": float("nan"), "bbox_area": float("nan")}
    valid = np.all(np.isfinite(center[:, :3]), axis=1)
    c = center[valid]
    if c.shape[0] < 2:
        return {"path_length": total, "displacement": float("nan"), "straightness": float("nan"), "direction_changes": float("nan"), "bbox_area": float("nan")}
    displacement = float(np.linalg.norm(c[-1, :3] - c[0, :3]))
    straightness = displacement / total if np.isfinite(total) and total > 1e-8 else float("nan")
    # Direction changes based on angle between consecutive velocity vectors.
    v = np.diff(c[:, :3], axis=0)
    norms = np.linalg.norm(v, axis=1)
    good = norms > 1e-8
    v = v[good]
    changes = 0
    if v.shape[0] >= 2:
        v1 = v[:-1]
        v2 = v[1:]
        denom = np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)
        cosang = np.sum(v1 * v2, axis=1) / np.maximum(denom, 1e-8)
        cosang = np.clip(cosang, -1.0, 1.0)
        angles = np.degrees(np.arccos(cosang))
        changes = int(np.sum(angles > 45.0))
    xy = c[:, :2]
    ranges = np.nanmax(xy, axis=0) - np.nanmin(xy, axis=0)
    bbox_area = float(ranges[0] * ranges[1]) if np.all(np.isfinite(ranges)) else float("nan")
    return {"path_length": total, "displacement": displacement, "straightness": straightness, "direction_changes": float(changes), "bbox_area": bbox_area}


def safe_sum(arr: np.ndarray) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.sum(arr)) if arr.size else float("nan")


def finite_distance(a: Optional[np.ndarray], b: Optional[np.ndarray]) -> np.ndarray:
    if a is None or b is None:
        return np.array([], dtype=np.float32)
    T = min(a.shape[0], b.shape[0])
    aa = a[:T, :3]
    bb = b[:T, :3]
    valid = np.all(np.isfinite(aa), axis=1) & np.all(np.isfinite(bb), axis=1)
    d = np.full(T, np.nan, dtype=np.float32)
    d[valid] = np.linalg.norm(aa[valid] - bb[valid], axis=1)
    return d


def resample_sequence(seq: np.ndarray, target_len: int) -> np.ndarray:
    seq = np.asarray(seq, dtype=np.float32)
    if seq.ndim != 2:
        return np.full((target_len, 1), np.nan, dtype=np.float32)
    T, D = seq.shape
    if T == 0:
        return np.full((target_len, D), np.nan, dtype=np.float32)
    if T == target_len:
        return seq.copy()
    x_old = np.linspace(0.0, 1.0, T)
    x_new = np.linspace(0.0, 1.0, target_len)
    out = np.full((target_len, D), np.nan, dtype=np.float32)
    for j in range(D):
        y = seq[:, j]
        valid = np.isfinite(y)
        if valid.sum() >= 2:
            out[:, j] = np.interp(x_new, x_old[valid], y[valid]).astype(np.float32)
        elif valid.sum() == 1:
            out[:, j] = float(y[valid][0])
    return out


def fill_nan_with_zero(seq: np.ndarray) -> np.ndarray:
    return np.nan_to_num(seq.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def combined_hand_trajectory(left: Optional[np.ndarray], right: Optional[np.ndarray], target_len: int) -> Optional[np.ndarray]:
    if left is None and right is None:
        return None
    T = 0
    if left is not None:
        T = max(T, left.shape[0])
    if right is not None:
        T = max(T, right.shape[0])
    if T == 0:
        return None
    l = left if left is not None else np.full((T, 3), np.nan, dtype=np.float32)
    r = right if right is not None else np.full((T, 3), np.nan, dtype=np.float32)
    T = min(l.shape[0], r.shape[0])
    traj = np.concatenate([l[:T, :3], r[:T, :3]], axis=1)
    traj = resample_sequence(traj, target_len)
    return fill_nan_with_zero(traj)




## 2. Trích đặc trưng chuyển động cho từng video


In [3]:
def extract_features_for_instance(
    seq: np.ndarray,
    parts: Dict[str, Optional[np.ndarray]],
    meta: Dict[str, Any],
    cfg: Dict[str, Any],
) -> Tuple[Dict[str, Any], Optional[np.ndarray]]:
    invalid_zero = bool(cfg.get("INVALID_ZERO_POINTS", True))
    fps = cfg.get("FPS", None)
    if fps in (None, "", "None"):
        fps = meta.get("source_fps", None)
    fps = None if fps in (None, "", "None") else float(fps)

    # Explicit parts override inferred parts. For flat seq, infer missing.
    inferred = infer_parts_from_flat(seq, cfg)
    merged: Dict[str, Optional[np.ndarray]] = {}
    for k in ["left_hand", "right_hand", "pose", "face"]:
        merged[k] = parts.get(k) if parts.get(k) is not None else inferred.get(k)

    left_center_raw, left_frame_missing, left_point_missing = center_from_landmarks(merged["left_hand"], invalid_zero)
    right_center_raw, right_frame_missing, right_point_missing = center_from_landmarks(merged["right_hand"], invalid_zero)
    face_center_raw, face_frame_missing, face_point_missing = center_from_landmarks(merged["face"], invalid_zero)
    pose_center_raw, pose_frame_missing, pose_point_missing = center_from_landmarks(merged["pose"], invalid_zero)

    if cfg.get("INTERPOLATE_MISSING_CENTERS", True):
        left_center_i = interpolate_centers(left_center_raw, cfg.get("MAX_INTERPOLATION_GAP", 8))
        right_center_i = interpolate_centers(right_center_raw, cfg.get("MAX_INTERPOLATION_GAP", 8))
        face_center_i = interpolate_centers(face_center_raw, cfg.get("MAX_INTERPOLATION_GAP", 8))
        pose_center_i = interpolate_centers(pose_center_raw, cfg.get("MAX_INTERPOLATION_GAP", 8))
    else:
        left_center_i, right_center_i, face_center_i, pose_center_i = left_center_raw, right_center_raw, face_center_raw, pose_center_raw

    body_center, scale = robust_body_scale(merged["pose"], left_center_i, right_center_i, invalid_zero)
    if body_center is None:
        body_center = pose_center_i
    body_center_i = interpolate_centers(body_center, cfg.get("MAX_INTERPOLATION_GAP", 8)) if cfg.get("INTERPOLATE_MISSING_CENTERS", True) else body_center

    do_norm = bool(cfg.get("NORMALIZE_BY_BODY_ANCHOR", True))
    left = normalize_center(left_center_i, body_center_i, scale, do_norm)
    right = normalize_center(right_center_i, body_center_i, scale, do_norm)
    face = normalize_center(face_center_i, body_center_i, scale, do_norm)
    body = normalize_center(body_center_i, body_center_i, scale, do_norm) if body_center_i is not None else None

    T = int(seq.shape[0])
    min_valid = int(cfg.get("MIN_VALID_HAND_FRAMES", 3))
    left_valid_frames = int(np.sum(np.all(np.isfinite(left_center_raw), axis=1))) if left_center_raw is not None else 0
    right_valid_frames = int(np.sum(np.all(np.isfinite(right_center_raw), axis=1))) if right_center_raw is not None else 0
    usable_motion = (left_valid_frames >= min_valid) or (right_valid_frames >= min_valid)

    left_motion = consecutive_distances(left)
    right_motion = consecutive_distances(right)
    # Sum motion of both hands per time step.
    max_len = max(left_motion.size, right_motion.size)
    if max_len:
        motion_per_frame = np.full(max_len, 0.0, dtype=np.float32)
        has_any = np.zeros(max_len, dtype=bool)
        for d in [left_motion, right_motion]:
            if d.size:
                valid = np.isfinite(d)
                motion_per_frame[:d.size][valid] += d[valid]
                has_any[:d.size] |= valid
        motion_per_frame = motion_per_frame.astype(np.float32)
        motion_per_frame[~has_any] = np.nan
    else:
        motion_per_frame = np.array([], dtype=np.float32)

    if fps is not None and fps > 0:
        velocity = motion_per_frame * fps
    else:
        velocity = motion_per_frame.copy()
    acceleration = np.diff(velocity)
    acceleration = acceleration[np.isfinite(acceleration)] if acceleration.size else acceleration

    left_stats = trajectory_stats(left)
    right_stats = trajectory_stats(right)
    traj_length = safe_sum(np.array([left_stats["path_length"], right_stats["path_length"]], dtype=float))
    displacement = safe_mean(np.array([left_stats["displacement"], right_stats["displacement"]], dtype=float))
    straightness = safe_mean(np.array([left_stats["straightness"], right_stats["straightness"]], dtype=float))
    direction_changes = safe_sum(np.array([left_stats["direction_changes"], right_stats["direction_changes"]], dtype=float))
    bbox_area = safe_mean(np.array([left_stats["bbox_area"], right_stats["bbox_area"]], dtype=float))

    # Dynamic distances.
    hand_face_vals = []
    hand_body_vals = []
    for hand in [left, right]:
        d_hf = finite_distance(hand, face)
        if d_hf.size:
            hand_face_vals.append(d_hf)
        d_hb = finite_distance(hand, body)
        if d_hb.size:
            hand_body_vals.append(d_hb)
    hand_face = np.concatenate(hand_face_vals) if hand_face_vals else np.array([], dtype=np.float32)
    hand_body = np.concatenate(hand_body_vals) if hand_body_vals else np.array([], dtype=np.float32)
    lr_dist = finite_distance(left, right)

    hand_missing_point_ratio = safe_mean(np.array([left_point_missing, right_point_missing], dtype=float))
    hand_missing_frame_ratio = safe_mean(np.array([left_frame_missing, right_frame_missing], dtype=float))

    row: Dict[str, Any] = {
        **meta,
        "num_frames": T,
        "feature_dim": int(seq.shape[1]),
        "usable_motion": bool(usable_motion),
        "normalization_scale": safe_float(scale),
        "has_left_hand": left_center_raw is not None,
        "has_right_hand": right_center_raw is not None,
        "has_pose": merged["pose"] is not None,
        "has_face": merged["face"] is not None,
        "left_valid_frames": left_valid_frames,
        "right_valid_frames": right_valid_frames,
        "left_frame_missing_ratio": left_frame_missing,
        "right_frame_missing_ratio": right_frame_missing,
        "hand_frame_missing_ratio": hand_missing_frame_ratio,
        "left_point_missing_ratio": left_point_missing,
        "right_point_missing_ratio": right_point_missing,
        "hand_point_missing_ratio": hand_missing_point_ratio,
        "pose_frame_missing_ratio": pose_frame_missing,
        "pose_point_missing_ratio": pose_point_missing,
        "face_frame_missing_ratio": face_frame_missing,
        "face_point_missing_ratio": face_point_missing,
        "total_motion": safe_sum(motion_per_frame),
        "mean_motion": safe_mean(motion_per_frame),
        "max_motion": safe_max(motion_per_frame),
        "motion_variance": float(np.nanvar(motion_per_frame)) if np.isfinite(motion_per_frame).any() else float("nan"),
        "mean_velocity": safe_mean(velocity),
        "max_velocity": safe_max(velocity),
        "velocity_std": safe_std(velocity),
        "mean_acceleration": safe_mean(np.abs(acceleration)),
        "max_acceleration": safe_max(np.abs(acceleration)),
        "acceleration_std": safe_std(acceleration),
        "trajectory_length": traj_length,
        "displacement": displacement,
        "straightness_ratio": straightness,
        "direction_change_count": direction_changes,
        "trajectory_bbox_area": bbox_area,
        "hand_face_dist_mean": safe_mean(hand_face),
        "hand_face_dist_std": safe_std(hand_face),
        "hand_body_dist_mean": safe_mean(hand_body),
        "hand_body_dist_std": safe_std(hand_body),
        "left_right_hand_dist_mean": safe_mean(lr_dist),
        "left_right_hand_dist_std": safe_std(lr_dist),
    }

    traj = combined_hand_trajectory(left, right, int(cfg.get("DTW_RESAMPLE_LEN", 30)))
    return row, traj




## 3. Quét `.npz` và tạo bảng feature


In [4]:
def iter_npz_instances(dataset_cfg: Dict[str, Any], cfg: Dict[str, Any]) -> Iterable[Tuple[Dict[str, Any], np.ndarray, Dict[str, Optional[np.ndarray]]]]:
    dataset = dataset_cfg["name"]
    root = Path(dataset_cfg["root"])
    label_map = load_label_map(dataset_cfg.get("label_map", ""))
    files = sorted(root.glob(cfg.get("NPZ_GLOB", "**/*.npz"))) if root.exists() else []
    if not root.exists():
        print(f"[WARN] Dataset root does not exist for {dataset}: {root}")
    if not files:
        print(f"[WARN] No .npz files found for {dataset} under {root}")
    for path in tqdm(files, desc=f"Scanning {dataset}"):
        try:
            with np.load(path, allow_pickle=True) as data:
                seq_key, seqs = find_sequence_array(data)
                if seqs is None:
                    keys = ",".join(list(data.keys()))
                    yield ({"dataset": dataset, "file": str(path), "error": f"no_sequence_array; keys={keys}"}, np.empty((0, 0)), {})
                    continue
                explicit_parts_all = get_explicit_parts(data, cfg)
                n_samples = int(seqs.shape[0])
                labels = labels_from_npz(data, n_samples, path, root, label_map)
                extra_meta = extraction_metadata_from_npz(data)
                video_name = str(extra_meta.get("video_name") or scalar_from_npz(data, "video_name", "") or "")
                for i in range(n_samples):
                    seq = seqs[i]
                    if seq.ndim != 2 or seq.shape[0] < 1 or seq.shape[1] < 1:
                        continue
                    sample_id_base = Path(video_name).stem if video_name else path.stem
                    sample_id = f"{sample_id_base}__sample_{i}" if n_samples > 1 else sample_id_base
                    try:
                        rel_file = str(path.relative_to(root))
                    except Exception:
                        rel_file = str(path)
                    meta = {
                        "dataset": dataset,
                        "label": str(labels[i]) if i < len(labels) else infer_label_from_path(path, root, i, label_map),
                        "file": rel_file,
                        "sample_id": sample_id,
                        "sample_index": i,
                        "npz_sequence_key": seq_key,
                    }
                    meta.update(extra_meta)
                    parts_i = {k: pick_sample_part(v, i, n_samples) for k, v in explicit_parts_all.items()}
                    yield meta, seq, parts_i
        except Exception as e:
            err_meta = {"dataset": dataset, "file": str(path), "error": f"load_failed: {e}"}
            yield err_meta, np.empty((0, 0)), {}


def inspect_npz_files(cfg: Dict[str, Any], dirs: Dict[str, Path], max_files_per_dataset: int = 5) -> None:
    rows = []
    for dcfg in cfg["DATASETS"]:
        dataset = dcfg["name"]
        root = Path(dcfg["root"])
        files = sorted(root.glob(cfg.get("NPZ_GLOB", "**/*.npz")))[:max_files_per_dataset] if root.exists() else []
        for path in files:
            try:
                with np.load(path, allow_pickle=True) as data:
                    for key in data.keys():
                        arr = data[key]
                        rows.append({
                            "dataset": dataset,
                            "file": str(path),
                            "key": key,
                            "shape": str(getattr(arr, "shape", None)),
                            "dtype": str(getattr(arr, "dtype", None)),
                        })
            except Exception as e:
                rows.append({"dataset": dataset, "file": str(path), "key": "ERROR", "shape": str(e), "dtype": ""})
    df = pd.DataFrame(rows)
    out = dirs["debug"] / "npz_structure_inspection.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved NPZ structure inspection: {out}")


def write_rows_checkpoint(rows: List[Dict[str, Any]], path: Path) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def process_all_features(cfg: Dict[str, Any], dirs: Dict[str, Path]) -> Tuple[pd.DataFrame, Optional[np.ndarray], List[str]]:
    out_csv = dirs["root"] / "motion_features_per_video.csv"
    traj_cache_path = dirs["cache"] / "motion_trajectories_resampled.npz"
    if out_csv.exists() and not cfg.get("FORCE_RECOMPUTE", True):
        print(f"[INFO] Using existing feature file: {out_csv}")
        df = pd.read_csv(out_csv)
        trajs, ids = None, []
        if traj_cache_path.exists():
            cache = np.load(traj_cache_path, allow_pickle=True)
            trajs = cache["trajectories"]
            ids = cache["sample_ids"].astype(str).tolist()
        return df, trajs, ids

    rows: List[Dict[str, Any]] = []
    trajs: List[np.ndarray] = []
    traj_ids: List[str] = []
    error_rows: List[Dict[str, Any]] = []
    checkpoint_every = int(cfg.get("CHECKPOINT_EVERY", 500))

    for dcfg in cfg["DATASETS"]:
        for item in iter_npz_instances(dcfg, cfg):
            meta, seq, parts = item
            if seq.size == 0 or "error" in meta:
                error_rows.append(meta)
                continue
            try:
                row, traj = extract_features_for_instance(seq, parts, meta, cfg)
                rows.append(row)
                if traj is not None and np.all(np.isfinite(traj)):
                    trajs.append(traj.astype(np.float32))
                    traj_ids.append(str(row["sample_id"]))
                if checkpoint_every > 0 and len(rows) % checkpoint_every == 0:
                    checkpoint = dirs["root"] / "motion_features_per_video_partial.csv"
                    write_rows_checkpoint(rows, checkpoint)
                    print(f"[checkpoint] {len(rows)} samples processed -> {checkpoint}")
            except Exception as e:
                meta2 = dict(meta)
                meta2["error"] = f"feature_failed: {e}"
                meta2["traceback"] = traceback.format_exc(limit=2)
                error_rows.append(meta2)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved per-video motion features: {out_csv} ({len(df)} rows)")

    if error_rows:
        err_df = pd.DataFrame(error_rows)
        err_path = dirs["debug"] / "failed_files.csv"
        err_df.to_csv(err_path, index=False, encoding="utf-8-sig")
        print(f"[WARN] Saved failed files/samples: {err_path} ({len(err_df)} rows)")

    traj_arr = np.stack(trajs, axis=0).astype(np.float32) if trajs else None
    if cfg.get("SAVE_TRAJECTORY_CACHE", True) and traj_arr is not None:
        np.savez_compressed(traj_cache_path, trajectories=traj_arr, sample_ids=np.array(traj_ids, dtype=object))
        print(f"[OK] Saved trajectory cache: {traj_cache_path} {traj_arr.shape}")
    return df, traj_arr, traj_ids




## 4. Tổng hợp chỉ số theo dataset/label


In [5]:
def summarize_dataset(df: pd.DataFrame, dirs: Dict[str, Path]) -> pd.DataFrame:
    metrics = [
        "num_frames", "hand_frame_missing_ratio", "hand_point_missing_ratio",
        "total_motion", "mean_motion", "max_motion", "motion_variance",
        "mean_velocity", "max_velocity", "velocity_std",
        "mean_acceleration", "max_acceleration", "acceleration_std",
        "trajectory_length", "displacement", "straightness_ratio",
        "direction_change_count", "trajectory_bbox_area",
        "hand_face_dist_mean", "hand_face_dist_std",
        "hand_body_dist_mean", "hand_body_dist_std",
        "left_right_hand_dist_mean", "left_right_hand_dist_std",
    ]
    rows = []
    for metric in metrics:
        if metric not in df.columns:
            continue
        for dataset, sub in df.groupby("dataset"):
            vals = pd.to_numeric(sub[metric], errors="coerce").dropna().values
            rows.append({
                "metric": metric,
                "dataset": dataset,
                "count": int(len(vals)),
                "mean": float(np.mean(vals)) if len(vals) else np.nan,
                "std": float(np.std(vals)) if len(vals) else np.nan,
                "median": float(np.median(vals)) if len(vals) else np.nan,
                "q1": float(np.quantile(vals, 0.25)) if len(vals) else np.nan,
                "q3": float(np.quantile(vals, 0.75)) if len(vals) else np.nan,
                "min": float(np.min(vals)) if len(vals) else np.nan,
                "max": float(np.max(vals)) if len(vals) else np.nan,
            })
    summary = pd.DataFrame(rows)
    out = dirs["root"] / "motion_summary_by_dataset.csv"
    summary.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved dataset summary: {out}")

    # Wide comparison table.
    if not summary.empty:
        wide = summary.pivot(index="metric", columns="dataset", values="mean").reset_index()
        wide_out = dirs["root"] / "motion_summary_by_dataset_wide.csv"
        wide.to_csv(wide_out, index=False, encoding="utf-8-sig")
        print(f"[OK] Saved wide dataset summary: {wide_out}")
    return summary


def summarize_label(df: pd.DataFrame, dirs: Dict[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Sequence length variance by label.
    rows = []
    for (dataset, label), sub in df.groupby(["dataset", "label"]):
        frames = pd.to_numeric(sub["num_frames"], errors="coerce").dropna().values
        rows.append({
            "dataset": dataset,
            "label": label,
            "num_samples": int(len(sub)),
            "mean_frames": float(np.mean(frames)) if len(frames) else np.nan,
            "std_frames": float(np.std(frames)) if len(frames) else np.nan,
            "min_frames": float(np.min(frames)) if len(frames) else np.nan,
            "max_frames": float(np.max(frames)) if len(frames) else np.nan,
            "sequence_length_variance": float(np.var(frames)) if len(frames) else np.nan,
        })
    seq_df = pd.DataFrame(rows)
    seq_out = dirs["root"] / "sequence_length_by_label.csv"
    seq_df.to_csv(seq_out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved sequence length by label: {seq_out}")

    agg_metrics = [
        "total_motion", "mean_velocity", "motion_variance", "mean_acceleration",
        "trajectory_length", "straightness_ratio", "direction_change_count",
        "left_right_hand_dist_mean", "hand_face_dist_mean", "hand_body_dist_mean"
    ]
    agg = df.groupby(["dataset", "label"]).agg(
        num_samples=("sample_id", "count"),
        mean_total_motion=("total_motion", "mean"),
        std_total_motion=("total_motion", "std"),
        mean_velocity=("mean_velocity", "mean"),
        motion_variance_mean=("motion_variance", "mean"),
        acceleration_mean=("mean_acceleration", "mean"),
        trajectory_length_mean=("trajectory_length", "mean"),
        straightness_mean=("straightness_ratio", "mean"),
        direction_change_mean=("direction_change_count", "mean"),
        lr_hand_dist_mean=("left_right_hand_dist_mean", "mean"),
        hand_face_dist_mean=("hand_face_dist_mean", "mean"),
        hand_body_dist_mean=("hand_body_dist_mean", "mean"),
        hand_missing_mean=("hand_frame_missing_ratio", "mean"),
    ).reset_index()
    agg = agg.merge(seq_df[["dataset", "label", "sequence_length_variance"]], on=["dataset", "label"], how="left")

    # Complexity score from normalized relevant metrics within each dataset.
    score_cols = ["mean_total_motion", "mean_velocity", "motion_variance_mean", "acceleration_mean", "trajectory_length_mean", "sequence_length_variance"]
    agg["motion_complexity_score"] = np.nan
    out_parts = []
    for dataset, sub in agg.groupby("dataset"):
        sub = sub.copy()
        score = np.zeros(len(sub), dtype=np.float32)
        used = 0
        for col in score_cols:
            vals = pd.to_numeric(sub[col], errors="coerce")
            if vals.notna().sum() >= 2:
                mn, mx = vals.min(), vals.max()
                if np.isfinite(mn) and np.isfinite(mx) and mx > mn:
                    norm = (vals - mn) / (mx - mn)
                    score += norm.fillna(0).values.astype(np.float32)
                    used += 1
        sub["motion_complexity_score"] = score / max(used, 1)
        out_parts.append(sub)
    agg = pd.concat(out_parts, axis=0) if out_parts else agg
    agg = agg.sort_values(["dataset", "motion_complexity_score"], ascending=[True, False])

    agg_out = dirs["root"] / "motion_features_by_label.csv"
    agg.to_csv(agg_out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved label-level motion features: {agg_out}")

    top_out = dirs["root"] / "top_complex_motion_labels.csv"
    agg.groupby("dataset", group_keys=False).head(50).to_csv(top_out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved top complex labels: {top_out}")
    return seq_df, agg




## 5. Vẽ biểu đồ


In [6]:
def plot_metric_box(df: pd.DataFrame, metric: str, dirs: Dict[str, Path]) -> None:
    if metric not in df.columns or df.empty:
        return
    data = []
    labels = []
    for dataset, sub in df.groupby("dataset"):
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna().values
        if len(vals):
            data.append(vals)
            labels.append(dataset)
    if not data:
        return
    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=labels, showfliers=False)
    plt.title(f"{metric}: VSL vs WLASL")
    plt.ylabel(metric)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    out = dirs["figures"] / f"boxplot_{metric}.png"
    plt.savefig(out, dpi=200)
    plt.close()


def plot_hist(df: pd.DataFrame, metric: str, dirs: Dict[str, Path]) -> None:
    if metric not in df.columns or df.empty:
        return
    plt.figure(figsize=(8, 5))
    for dataset, sub in df.groupby("dataset"):
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna().values
        if len(vals):
            plt.hist(vals, bins=40, alpha=0.5, label=dataset)
    plt.title(f"Distribution of {metric}")
    plt.xlabel(metric)
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    out = dirs["figures"] / f"histogram_{metric}.png"
    plt.savefig(out, dpi=200)
    plt.close()


def plot_scatter(df: pd.DataFrame, x: str, y: str, dirs: Dict[str, Path]) -> None:
    if x not in df.columns or y not in df.columns or df.empty:
        return
    plt.figure(figsize=(8, 5))
    for dataset, sub in df.groupby("dataset"):
        xx = pd.to_numeric(sub[x], errors="coerce")
        yy = pd.to_numeric(sub[y], errors="coerce")
        mask = xx.notna() & yy.notna()
        if mask.sum():
            plt.scatter(xx[mask], yy[mask], s=10, alpha=0.5, label=dataset)
    plt.title(f"{x} vs {y}")
    plt.xlabel(x)
    plt.ylabel(y)
    plt.legend()
    plt.tight_layout()
    out = dirs["figures"] / f"scatter_{x}_vs_{y}.png"
    plt.savefig(out, dpi=200)
    plt.close()


def plot_top_labels(label_df: pd.DataFrame, cfg: Dict[str, Any], dirs: Dict[str, Path]) -> None:
    if label_df.empty or "motion_complexity_score" not in label_df.columns:
        return
    top_n = int(cfg.get("PLOT_TOP_N_LABELS", 30))
    for dataset, sub in label_df.groupby("dataset"):
        sub = sub.sort_values("motion_complexity_score", ascending=False).head(top_n)
        if sub.empty:
            continue
        plt.figure(figsize=(10, max(5, 0.25 * len(sub))))
        y = np.arange(len(sub))
        plt.barh(y, sub["motion_complexity_score"].values)
        plt.yticks(y, sub["label"].astype(str).values)
        plt.gca().invert_yaxis()
        plt.xlabel("motion_complexity_score")
        plt.title(f"Top {top_n} complex-motion labels - {dataset}")
        plt.tight_layout()
        out = dirs["figures"] / f"top_complex_motion_labels_{dataset}.png"
        plt.savefig(out, dpi=200)
        plt.close()


def plot_trajectory_examples(trajs: Optional[np.ndarray], traj_ids: List[str], df: pd.DataFrame, cfg: Dict[str, Any], dirs: Dict[str, Path]) -> None:
    if trajs is None or len(traj_ids) == 0 or df.empty:
        return
    id_to_idx = {sid: i for i, sid in enumerate(traj_ids)}
    n = int(cfg.get("TRAJECTORY_EXAMPLES_PER_DATASET", 6))
    for dataset, sub in df.groupby("dataset"):
        sub = sub.sort_values("total_motion", ascending=False).head(n)
        plt.figure(figsize=(8, 6))
        plotted = 0
        for _, row in sub.iterrows():
            sid = str(row["sample_id"])
            if sid not in id_to_idx:
                continue
            tr = trajs[id_to_idx[sid]]
            # left hand x,y = cols 0,1; right hand x,y = cols 3,4.
            plt.plot(tr[:, 0], tr[:, 1], alpha=0.8, label=f"{row['label']} L")
            plt.plot(tr[:, 3], tr[:, 4], linestyle="--", alpha=0.8, label=f"{row['label']} R")
            plotted += 1
        if plotted == 0:
            plt.close()
            continue
        plt.title(f"Hand trajectory examples - {dataset}")
        plt.xlabel("normalized x")
        plt.ylabel("normalized y")
        plt.legend(fontsize=7, loc="best")
        plt.tight_layout()
        out = dirs["figures"] / f"trajectory_examples_{dataset}.png"
        plt.savefig(out, dpi=200)
        plt.close()


def generate_plots(df: pd.DataFrame, label_df: pd.DataFrame, trajs: Optional[np.ndarray], traj_ids: List[str], cfg: Dict[str, Any], dirs: Dict[str, Path]) -> None:
    for metric in [
        "total_motion", "mean_velocity", "motion_variance", "mean_acceleration",
        "trajectory_length", "straightness_ratio", "left_right_hand_dist_mean", "num_frames"
    ]:
        plot_metric_box(df, metric, dirs)
    for metric in ["num_frames", "total_motion", "mean_velocity", "motion_variance"]:
        plot_hist(df, metric, dirs)
    plot_scatter(df, "total_motion", "motion_variance", dirs)
    plot_scatter(df, "trajectory_length", "straightness_ratio", dirs)
    plot_top_labels(label_df, cfg, dirs)
    plot_trajectory_examples(trajs, traj_ids, df, cfg, dirs)
    print(f"[OK] Saved figures to: {dirs['figures']}")


# =========================


## 6. DTW, report và hàm chạy chính


In [7]:
# DTW analysis
# =========================

def dtw_distance(a: np.ndarray, b: np.ndarray, window: Optional[int] = None) -> float:
    """DTW distance for two sequences (T,D), with optional Sakoe-Chiba window."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    n, m = a.shape[0], b.shape[0]
    if n == 0 or m == 0:
        return float("nan")
    if window is None:
        window = max(n, m)
    window = max(int(window), abs(n - m))
    inf = float("inf")
    prev = np.full(m + 1, inf, dtype=np.float32)
    curr = np.full(m + 1, inf, dtype=np.float32)
    prev[0] = 0.0
    for i in range(1, n + 1):
        curr[:] = inf
        j_start = max(1, i - window)
        j_end = min(m, i + window)
        ai = a[i - 1]
        for j in range(j_start, j_end + 1):
            cost = float(np.linalg.norm(ai - b[j - 1]))
            curr[j] = cost + min(prev[j], curr[j - 1], prev[j - 1])
        prev, curr = curr, prev
    return float(prev[m] / (n + m))


def run_intra_class_dtw(df: pd.DataFrame, trajs: Optional[np.ndarray], traj_ids: List[str], cfg: Dict[str, Any], dirs: Dict[str, Path]) -> pd.DataFrame:
    if trajs is None or len(traj_ids) == 0 or df.empty:
        return pd.DataFrame()
    id_to_idx = {sid: i for i, sid in enumerate(traj_ids)}
    min_samples = int(cfg.get("DTW_MIN_SAMPLES_PER_LABEL", 2))
    max_pairs = cfg.get("DTW_MAX_PAIRS_PER_LABEL", None)
    max_pairs = None if max_pairs in (None, "", "None") else int(max_pairs)
    rng = np.random.default_rng(int(cfg.get("DTW_RANDOM_SEED", 42)))
    window = cfg.get("DTW_WINDOW", None)

    rows = []
    pair_rows = []
    groups = list(df.groupby(["dataset", "label"]))
    for (dataset, label), sub in tqdm(groups, desc="Intra-class DTW"):
        ids = [str(x) for x in sub["sample_id"].tolist() if str(x) in id_to_idx]
        if len(ids) < min_samples:
            continue
        pairs = [(i, j) for i in range(len(ids)) for j in range(i + 1, len(ids))]
        if max_pairs is not None and len(pairs) > max_pairs:
            selected = rng.choice(len(pairs), size=max_pairs, replace=False)
            pairs = [pairs[k] for k in selected]
        vals = []
        for i, j in pairs:
            id_a, id_b = ids[i], ids[j]
            d = dtw_distance(trajs[id_to_idx[id_a]], trajs[id_to_idx[id_b]], window=window)
            if np.isfinite(d):
                vals.append(d)
                if len(pair_rows) < 200000:  # keep file size reasonable
                    pair_rows.append({"dataset": dataset, "label": label, "sample_id_a": id_a, "sample_id_b": id_b, "dtw_distance": d})
        if vals:
            vals_arr = np.array(vals, dtype=float)
            rows.append({
                "dataset": dataset,
                "label": label,
                "num_samples": len(ids),
                "num_pairs": len(vals),
                "intra_class_dtw_mean": float(np.mean(vals_arr)),
                "intra_class_dtw_std": float(np.std(vals_arr)),
                "intra_class_dtw_min": float(np.min(vals_arr)),
                "intra_class_dtw_max": float(np.max(vals_arr)),
                "intra_class_dtw_median": float(np.median(vals_arr)),
            })
    out_df = pd.DataFrame(rows).sort_values(["dataset", "intra_class_dtw_mean"], ascending=[True, False]) if rows else pd.DataFrame()
    out = dirs["dtw"] / "dtw_intra_class_summary.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved intra-class DTW summary: {out}")
    if pair_rows:
        pair_df = pd.DataFrame(pair_rows)
        pair_out = dirs["dtw"] / "dtw_intra_class_pairs_sample.csv"
        pair_df.to_csv(pair_out, index=False, encoding="utf-8-sig")
        print(f"[OK] Saved intra-class DTW pair sample: {pair_out}")
    return out_df


def compute_label_prototypes(df: pd.DataFrame, trajs: Optional[np.ndarray], traj_ids: List[str]) -> Tuple[pd.DataFrame, Optional[np.ndarray]]:
    if trajs is None or len(traj_ids) == 0 or df.empty:
        return pd.DataFrame(), None
    id_to_idx = {sid: i for i, sid in enumerate(traj_ids)}
    rows = []
    proto_list = []
    for (dataset, label), sub in df.groupby(["dataset", "label"]):
        indices = [id_to_idx[str(sid)] for sid in sub["sample_id"].tolist() if str(sid) in id_to_idx]
        if not indices:
            continue
        proto = np.mean(trajs[indices], axis=0).astype(np.float32)
        proto_list.append(proto)
        rows.append({"dataset": dataset, "label": label, "num_samples": len(indices), "prototype_index": len(proto_list) - 1})
    if not proto_list:
        return pd.DataFrame(), None
    return pd.DataFrame(rows), np.stack(proto_list, axis=0).astype(np.float32)


def run_inter_class_dtw(df: pd.DataFrame, trajs: Optional[np.ndarray], traj_ids: List[str], cfg: Dict[str, Any], dirs: Dict[str, Path]) -> pd.DataFrame:
    proto_df, protos = compute_label_prototypes(df, trajs, traj_ids)
    if protos is None or proto_df.empty:
        return pd.DataFrame()
    exhaustive = bool(cfg.get("INTER_CLASS_DTW_EXHAUSTIVE", False))
    max_exhaustive = int(cfg.get("INTER_CLASS_MAX_EXHAUSTIVE_LABELS", 600))
    candidates_per_label = int(cfg.get("INTER_CLASS_CANDIDATES_PER_LABEL", 20))
    window = cfg.get("DTW_WINDOW", None)
    rows = []

    # Save prototypes metadata.
    proto_meta_out = dirs["dtw"] / "dtw_label_prototypes.csv"
    proto_df.to_csv(proto_meta_out, index=False, encoding="utf-8-sig")

    for dataset, sub_meta in proto_df.groupby("dataset"):
        idxs = sub_meta["prototype_index"].values.astype(int)
        labels = sub_meta["label"].astype(str).values
        P = protos[idxs]
        n = P.shape[0]
        if n < 2:
            continue
        flat = P.reshape(n, -1)
        # Standardize to reduce scale dominance.
        mu = np.mean(flat, axis=0, keepdims=True)
        sd = np.std(flat, axis=0, keepdims=True) + 1e-6
        flat_z = (flat - mu) / sd

        if exhaustive and n <= max_exhaustive:
            candidate_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
        else:
            # Use all labels but only top nearest candidates by Euclidean prefilter.
            candidate_set = set()
            # Compute distances in chunks to avoid huge memory.
            chunk = 512
            for start in tqdm(range(0, n, chunk), desc=f"Inter-class candidate search {dataset}"):
                end = min(n, start + chunk)
                # squared Euclidean: (a-b)^2 = a^2 + b^2 - 2ab
                A = flat_z[start:end]
                d2 = np.sum(A * A, axis=1, keepdims=True) + np.sum(flat_z * flat_z, axis=1)[None, :] - 2 * A @ flat_z.T
                d2[:, start:end] += np.eye(end - start, dtype=np.float32) * 1e12
                k = min(candidates_per_label, n - 1)
                nn = np.argpartition(d2, kth=k, axis=1)[:, :k]
                for local_i, neighs in enumerate(nn):
                    i = start + local_i
                    for j in neighs.tolist():
                        if i == j:
                            continue
                        a, b = (i, j) if i < j else (j, i)
                        candidate_set.add((a, b))
            candidate_pairs = sorted(candidate_set)

        for i, j in tqdm(candidate_pairs, desc=f"Inter-class DTW {dataset}"):
            d = dtw_distance(P[i], P[j], window=window)
            if np.isfinite(d):
                rows.append({
                    "dataset": dataset,
                    "label_a": labels[i],
                    "label_b": labels[j],
                    "dtw_distance": d,
                    "mode": "exhaustive" if exhaustive and n <= max_exhaustive else "nearest_candidates_then_dtw",
                })

    out_df = pd.DataFrame(rows)
    if not out_df.empty:
        out_df = out_df.sort_values(["dataset", "dtw_distance"], ascending=[True, True])
    out = dirs["dtw"] / "dtw_inter_class_candidate_pairs.csv"
    out_df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved inter-class DTW candidate pairs: {out}")
    return out_df


def plot_dtw_outputs(intra_df: pd.DataFrame, inter_df: pd.DataFrame, dirs: Dict[str, Path]) -> None:
    if intra_df is not None and not intra_df.empty:
        plt.figure(figsize=(8, 5))
        for dataset, sub in intra_df.groupby("dataset"):
            vals = pd.to_numeric(sub["intra_class_dtw_mean"], errors="coerce").dropna().values
            if len(vals):
                plt.hist(vals, bins=40, alpha=0.5, label=dataset)
        plt.xlabel("intra_class_dtw_mean")
        plt.ylabel("count")
        plt.title("Distribution of intra-class DTW")
        plt.legend()
        plt.tight_layout()
        out = dirs["figures"] / "histogram_intra_class_dtw.png"
        plt.savefig(out, dpi=200)
        plt.close()
    if inter_df is not None and not inter_df.empty:
        for dataset, sub in inter_df.groupby("dataset"):
            top = sub.sort_values("dtw_distance", ascending=True).head(30)
            if top.empty:
                continue
            labels = (top["label_a"].astype(str) + " ↔ " + top["label_b"].astype(str)).values
            plt.figure(figsize=(10, max(5, 0.25 * len(top))))
            y = np.arange(len(top))
            plt.barh(y, top["dtw_distance"].values)
            plt.yticks(y, labels)
            plt.gca().invert_yaxis()
            plt.xlabel("DTW distance, lower = more similar trajectory")
            plt.title(f"Top similar inter-class trajectory pairs - {dataset}")
            plt.tight_layout()
            out = dirs["figures"] / f"top_inter_class_dtw_pairs_{dataset}.png"
            plt.savefig(out, dpi=200)
            plt.close()


def generate_markdown_report(df: pd.DataFrame, summary: pd.DataFrame, label_df: pd.DataFrame, intra_df: pd.DataFrame, inter_df: pd.DataFrame, cfg: Dict[str, Any], dirs: Dict[str, Path]) -> None:
    report_path = dirs["root"] / "README_RESULTS_4_7.md"
    lines = []
    lines.append("# Section 4.7 - Motion Feature Analysis Results\n")
    lines.append("## Files generated\n")
    lines.append("- `motion_features_per_video.csv`: one row per video/sample.\n")
    lines.append("- `motion_summary_by_dataset.csv`: dataset-level long summary.\n")
    lines.append("- `motion_summary_by_dataset_wide.csv`: dataset-level mean comparison table.\n")
    lines.append("- `sequence_length_by_label.csv`: sequence length statistics per label.\n")
    lines.append("- `motion_features_by_label.csv`: label-level motion statistics.\n")
    lines.append("- `top_complex_motion_labels.csv`: labels with highest motion complexity score.\n")
    lines.append("- `dtw/dtw_intra_class_summary.csv`: intra-class trajectory variability.\n")
    lines.append("- `dtw/dtw_inter_class_candidate_pairs.csv`: likely confusing inter-class trajectory pairs.\n")
    lines.append("- `figures/`: plots for comparison and reporting.\n")
    lines.append("\n## How to interpret key metrics\n")
    lines.append("- `total_motion`: total hand displacement; higher means larger movement amplitude.\n")
    lines.append("- `mean_velocity`: average per-frame movement speed; higher means faster signing.\n")
    lines.append("- `motion_variance`: movement stability; higher means more irregular motion.\n")
    lines.append("- `sequence_length_variance`: temporal consistency within a label; higher means samples of the same sign have different durations.\n")
    lines.append("- `trajectory_length`: total path length of both hands.\n")
    lines.append("- `straightness_ratio`: close to 1 = straight path; lower = curved/repeated/complex path.\n")
    lines.append("- `left_right_hand_dist_mean`: average distance between two hands; useful for two-hand interaction analysis.\n")
    lines.append("- `intra_class_dtw_mean`: same-label movement variability; higher means signers perform the same label differently.\n")
    lines.append("- `inter_class dtw_distance`: lower means different labels have similar trajectories and may be confused.\n")
    lines.append("\n## Quick dataset comparison\n")
    if not summary.empty:
        try:
            pivot = summary.pivot(index="metric", columns="dataset", values="mean")
            for metric in ["total_motion", "mean_velocity", "motion_variance", "trajectory_length", "straightness_ratio", "num_frames"]:
                if metric in pivot.index:
                    vals = pivot.loc[metric].dropna()
                    if len(vals) >= 2:
                        higher = vals.idxmax()
                        lines.append(f"- `{metric}` is higher on average in **{higher}**.\n")
        except Exception:
            pass
    lines.append("\n## Suggested conclusion template\n")
    lines.append("> Based on the motion feature analysis, the dataset with higher total motion, velocity, motion variance, and intra-class DTW can be considered more temporally complex. High sequence length variance indicates inconsistent sign duration within the same label, while low inter-class DTW reveals pairs of signs with similar movement trajectories that may cause recognition confusion.\n")
    report_path.write_text("".join(lines), encoding="utf-8")
    print(f"[OK] Saved result README: {report_path}")


def main(config: Optional[Any] = None) -> Dict[str, Any]:
    cfg = load_config(config)
    dirs = ensure_dirs(Path(cfg["OUTPUT_DIR"]))
    print("===== Section 4.7 Motion Feature Analysis =====")
    print(f"Output directory: {dirs['root']}")
    for d in cfg["DATASETS"]:
        print(f"Dataset {d['name']}: {d['root']}")

    inspect_npz_files(cfg, dirs)
    df, trajs, traj_ids = process_all_features(cfg, dirs)
    if df.empty:
        print("[STOP] No valid samples processed. Check dataset paths and .npz structure.")
        return {"config": cfg, "dirs": dirs, "features": df}

    summary = summarize_dataset(df, dirs)
    seq_df, label_df = summarize_label(df, dirs)
    generate_plots(df, label_df, trajs, traj_ids, cfg, dirs)

    intra_df = pd.DataFrame()
    inter_df = pd.DataFrame()
    if cfg.get("RUN_DTW", True):
        intra_df = run_intra_class_dtw(df, trajs, traj_ids, cfg, dirs)
        if cfg.get("RUN_INTER_CLASS_DTW", True):
            inter_df = run_inter_class_dtw(df, trajs, traj_ids, cfg, dirs)
        plot_dtw_outputs(intra_df, inter_df, dirs)

    generate_markdown_report(df, summary, label_df, intra_df, inter_df, cfg, dirs)
    print("===== DONE 4.7 =====")
    return {
        "config": cfg,
        "dirs": dirs,
        "features": df,
        "summary": summary,
        "sequence_length_by_label": seq_df,
        "label_features": label_df,
        "intra_dtw": intra_df,
        "inter_dtw": inter_df,
        "trajectories": trajs,
        "trajectory_ids": traj_ids,
    }


## 7. Kiểm tra nhanh dữ liệu đầu vào

Cell này kiểm tra thư mục ASL/VSL có tồn tại, đếm số file `.npz`, và in keys của file đầu tiên để xác nhận đúng format trích xuất.


In [8]:
from pathlib import Path
import numpy as np

for d in CONFIG['DATASETS']:
    root = Path(d['root'])
    files = sorted(root.glob(CONFIG.get('NPZ_GLOB', '**/*.npz'))) if root.exists() else []
    print(d['name'], 'root_exists=', root.exists(), 'npz_files=', len(files), 'root=', root)
    for p in files[:3]:
        print('   ', p)
    if files:
        with np.load(files[0], allow_pickle=True) as data:
            print('   sample_keys=', list(data.keys()))
            for key in ['label', 'video_name', 'pose', 'hands', 'face', 'mouth', 'valid_mask']:
                if key in data.keys():
                    arr = data[key]
                    print('   ', key, 'shape=', getattr(arr, 'shape', None), 'dtype=', getattr(arr, 'dtype', None))


ASL root_exists= True npz_files= 11980 root= /content/drive/MyDrive/data/ASL
    /content/drive/MyDrive/data/ASL/a/01610.npz
    /content/drive/MyDrive/data/ASL/a/01612.npz
    /content/drive/MyDrive/data/ASL/a/01615.npz
   sample_keys= ['label', 'video_name', 'target_frames', 'source_fps', 'source_total_frames', 'sample_indices', 'train_feature_dim', 'pose', 'hands', 'face', 'mouth', 'valid_mask']
    label shape= () dtype= <U1
    video_name shape= () dtype= <U9
    pose shape= (60, 330) dtype= float16
    hands shape= (60, 252) dtype= float16
    face shape= (60, 52) dtype= float16
    mouth shape= (60, 36) dtype= float16
    valid_mask shape= (60, 4) dtype= uint8
VSL root_exists= True npz_files= 4362 root= /content/drive/MyDrive/data/VSL
    /content/drive/MyDrive/data/VSL/0 (số không)/D0529.npz
    /content/drive/MyDrive/data/VSL/1/D0530.npz
    /content/drive/MyDrive/data/VSL/1 000 000 (một triệu)/D0551B.npz
   sample_keys= ['label', 'video_name', 'target_frames', 'source_

## 8. Chạy phân tích 4.7

Cell này tạo toàn bộ CSV, biểu đồ và DTW trong `OUTPUT_DIR`.


In [9]:
RESULTS = main(CONFIG)


===== Section 4.7 Motion Feature Analysis =====
Output directory: /content/drive/MyDrive/data/output_4.7
Dataset ASL: /content/drive/MyDrive/data/ASL
Dataset VSL: /content/drive/MyDrive/data/VSL
[OK] Saved NPZ structure inspection: /content/drive/MyDrive/data/output_4.7/debug/npz_structure_inspection.csv


Scanning ASL:   0%|          | 0/11980 [00:00<?, ?it/s]

[checkpoint] 500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 1000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 1500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 2000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 2500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 3000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 3500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 4000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 4500 samples processed -> /content/drive/MyDrive/data/output

Scanning VSL:   0%|          | 0/4362 [00:00<?, ?it/s]

[checkpoint] 12000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 12500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 13000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 13500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 14000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 14500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 15000 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 15500 samples processed -> /content/drive/MyDrive/data/output_4.7/motion_features_per_video_partial.csv
[checkpoint] 16000 samples processed -> /content/drive/MyDrive/d

...

Inter-class candidate search ASL:   0%|          | 0/4 [00:00<?, ?it/s]

Inter-class DTW ASL:   0%|          | 0/30873 [00:00<?, ?it/s]

Inter-class candidate search VSL:   0%|          | 0/7 [00:00<?, ?it/s]

Inter-class DTW VSL:   0%|          | 0/54767 [00:00<?, ?it/s]

[OK] Saved inter-class DTW candidate pairs: /content/drive/MyDrive/data/output_4.7/dtw/dtw_inter_class_candidate_pairs.csv
[OK] Saved result README: /content/drive/MyDrive/data/output_4.7/README_RESULTS_4_7.md
===== DONE 4.7 =====


## 9. Xem danh sách file kết quả


In [10]:
from pathlib import Path

out = Path(CONFIG['OUTPUT_DIR'])
print('OUTPUT_DIR:', out)
if out.exists():
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(p)
else:
    print('Ch?a c? output. Ki?m tra l?i ???ng d?n dataset ho?c l?i ? cell ch?y ph?n t?ch.')


OUTPUT_DIR: /content/drive/MyDrive/data/output_4.7
/content/drive/MyDrive/data/output_4.7/README_RESULTS_4_7.md
/content/drive/MyDrive/data/output_4.7/cache/motion_trajectories_resampled.npz
/content/drive/MyDrive/data/output_4.7/debug/npz_structure_inspection.csv
/content/drive/MyDrive/data/output_4.7/dtw/dtw_inter_class_candidate_pairs.csv
/content/drive/MyDrive/data/output_4.7/dtw/dtw_intra_class_pairs_sample.csv
/content/drive/MyDrive/data/output_4.7/dtw/dtw_intra_class_summary.csv
/content/drive/MyDrive/data/output_4.7/dtw/dtw_label_prototypes.csv
/content/drive/MyDrive/data/output_4.7/figures/boxplot_left_right_hand_dist_mean.png
/content/drive/MyDrive/data/output_4.7/figures/boxplot_mean_acceleration.png
/content/drive/MyDrive/data/output_4.7/figures/boxplot_mean_velocity.png
/content/drive/MyDrive/data/output_4.7/figures/boxplot_motion_variance.png
/content/drive/MyDrive/data/output_4.7/figures/boxplot_num_frames.png
/content/drive/MyDrive/data/output_4.7/figures/boxplot_straig

## 10. Xem nhanh bảng tổng hợp


In [11]:
import pandas as pd
from pathlib import Path

out = Path(CONFIG['OUTPUT_DIR'])
summary_path = out / 'motion_summary_by_dataset_wide.csv'
top_path = out / 'top_complex_motion_labels.csv'
features_path = out / 'motion_features_per_video.csv'

if summary_path.exists():
    display(pd.read_csv(summary_path).head(50))
if top_path.exists():
    display(pd.read_csv(top_path).head(50))
if features_path.exists():
    display(pd.read_csv(features_path).head(20))


,metric,ASL,VSL
0,acceleration_std,7.519970,9.122484
1,direction_change_count,40.790028,67.250631
2,displacement,0.773077,0.415492
3,hand_body_dist_mean,2.212182,2.488008
4,hand_body_dist_std,0.489070,1.092717
5,hand_face_dist_mean,2.399594,2.881762
6,hand_face_dist_std,0.358527,0.672662
7,hand_frame_missing_ratio,0.396300,0.322581
8,hand_point_missing_ratio,0.396300,0.322581
9,left_right_hand_dist_mean,1.452657,1.973124


,dataset,label,num_samples,mean_total_motion,std_total_motion,mean_velocity,motion_variance_mean,acceleration_mean,trajectory_length_mean,straightness_mean,direction_change_mean,lr_hand_dist_mean,hand_face_dist_mean,hand_body_dist_mean,hand_missing_mean,sequence_length_variance,motion_complexity_score
0,ASL,ceiling,6,33.276389,13.014543,18.096695,0.415195,12.679056,33.276389,0.064818,44.666667,1.558910,2.101073,2.612487,0.411715,165.583333,0.765440
1,ASL,shampoo,5,37.663288,15.249222,17.438929,0.280538,14.291261,37.663289,0.057037,56.250000,1.543373,1.834965,2.434857,0.381228,96.640000,0.762431
2,ASL,climb,5,32.801574,6.386780,15.887647,0.179118,10.350163,32.801574,0.114763,64.400000,1.561756,1.977724,2.148933,0.314413,1181.760000,0.676670
3,ASL,swimming,6,29.759486,20.597942,13.601718,0.416633,9.166733,29.759486,0.067967,60.833333,1.481941,2.475212,2.126554,0.300608,219.222222,0.645049
4,ASL,stadium,6,27.436633,12.460566,14.154674,0.391426,12.001726,27.436632,0.100302,53.500000,1.748456,2.269366,2.262422,0.230971,32.138889,0.641438
5,ASL,binoculars,4,26.324512,11.390063,11.921332,0.351467,15.486889,26.324512,0.096747,37.750000,1.318839,1.782939,2.352433,0.133333,0.000000,0.632648
6,ASL,curtain,2,27.070765,14.474310,13.866697,0.296895,14.802032,27.070765,0.124266,47.000000,1.479987,2.679207,2.852487,0.137500,0.000000,0.629309
7,ASL,helmet,5,28.535227,3.141061,15.507408,0.248792,12.149717,28.535227,0.093513,57.200000,1.360963,1.963194,2.326284,0.321218,277.760000,0.625350
8,ASL,owl,5,28.867226,6.876177,15.622663,0.259223,12.134582,28.867226,0.064928,54.800000,1.373094,1.795549,2.268268,0.312117,56.640000,0.620003
9,ASL,visualize,5,29.409638,20.719452,12.642966,0.279461,11.065346,29.409638,0.108209,49.800000,1.644860,2.293816,2.894752,0.382517,391.360000,0.610388


,dataset,label,file,sample_id,sample_index,npz_sequence_key,npz_format,video_name,target_frames,source_fps,...,displacement,straightness_ratio,direction_change_count,trajectory_bbox_area,hand_face_dist_mean,hand_face_dist_std,hand_body_dist_mean,hand_body_dist_std,left_right_hand_dist_mean,left_right_hand_dist_std
0,ASL,a,a/01610.npz,01610,0,pose+hands+face+mouth,asl_vsl_extractor_v2,01610.mp4,60,30.000000,...,0.613480,0.074342,30.0,0.395665,2.222715,0.362160,3.141090,0.246587,NaN,NaN
1,ASL,a,a/01612.npz,01612,0,pose+hands+face+mouth,asl_vsl_extractor_v2,01612.mp4,60,29.969999,...,0.277718,0.205679,18.0,0.002352,2.890307,0.053431,3.915429,0.086245,NaN,NaN
2,ASL,a,a/01615.npz,01615,0,pose+hands+face+mouth,asl_vsl_extractor_v2,01615.mp4,60,30.331450,...,0.636408,0.059084,22.0,0.238286,2.576676,0.192673,3.108938,0.553716,NaN,NaN
3,ASL,a,a/66039.npz,66039,0,pose+hands+face+mouth,asl_vsl_extractor_v2,66039.mp4,60,23.976076,...,1.390160,0.175867,14.0,0.199671,2.954763,0.186120,3.316186,0.660347,NaN,NaN
4,ASL,a lot,a lot/02124.npz,02124,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02124.mp4,60,25.000000,...,0.342325,0.049531,46.0,0.192623,2.069776,0.208224,1.663582,0.367053,1.102719,0.174292
5,ASL,a lot,a lot/02125.npz,02125,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02125.mp4,60,23.976109,...,0.851496,0.116133,33.0,0.519501,2.580242,0.367051,1.617801,0.438082,1.589123,0.389213
6,ASL,a lot,a lot/02126.npz,02126,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02126.mp4,60,29.996876,...,0.568692,0.070968,64.0,0.374858,1.913549,0.377489,1.884508,0.499849,1.440837,0.321946
7,ASL,a lot,a lot/02128.npz,02128,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02128.mp4,60,29.969999,...,0.444539,0.302660,16.0,0.057882,2.289966,0.204883,1.891251,0.385222,1.465402,0.097418
8,ASL,a lot,a lot/02129.npz,02129,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02129.mp4,60,29.969999,...,0.505190,0.584523,11.0,0.026484,2.283124,0.095497,1.748622,0.168248,0.900056,0.026426
9,ASL,a lot,a lot/02130.npz,02130,0,pose+hands+face+mouth,asl_vsl_extractor_v2,02130.mp4,60,29.969999,...,0.307643,0.122233,36.0,0.019176,2.385477,0.094385,2.103131,0.213431,0.966340,0.048558
